# **ESTIMACIÓN DE PARÁMETROS — MODELO DIXON-COLES**

Este notebook ajusta los parámetros del modelo Dixon-Coles (fuerza de ataque/defensa por equipo, ventaja de local `home_adv` y corrección `rho`) por máxima verosimilitud, a partir de los resultados históricos descargados de football-data.co.uk. El resultado se guarda como JSON y es el que consume `prediccion_premier_league.ipynb`.

Las funciones se definen una sola vez; después se ejecutan dos veces con distinto rango de temporadas de entrenamiento/test, una por cada conjunto de parámetros que usa el notebook principal (24/25 y 25/26).

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
from scipy.stats import poisson
from scipy.optimize import minimize

warnings.filterwarnings('ignore')


def download_season(season_code):
    url = f"https://www.football-data.co.uk/mmz4281/{season_code}/E0.csv"
    try:
        df = pd.read_csv(url)
        df['Season'] = season_code
        print(f"  ✓ Temporada {season_code}: {len(df)} partidos")
        return df
    except Exception as e:
        print(f"  ✗ Error descargando {season_code}: {e}")
        return None

def load_seasons(season_codes):
    dfs = [df for code in season_codes if (df := download_season(code)) is not None]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_data(df):
    df = df[['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Date', 'Season']].copy()
    df = df.rename(columns={'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})
    df = df.dropna(subset=['HomeGoals', 'AwayGoals'])
    df['HomeGoals'] = df['HomeGoals'].astype(int)
    df['AwayGoals'] = df['AwayGoals'].astype(int)
    for fmt in ['%d/%m/%Y', '%d/%m/%y']:
        try:
            df['Date'] = pd.to_datetime(df['Date'], format=fmt)
            break
        except Exception:
            pass
    return df

def rho_correction(x, y, lam, mu, rho):
    if   x == 0 and y == 0: return 1 - lam * mu * rho
    elif x == 0 and y == 1: return 1 + lam * rho
    elif x == 1 and y == 0: return 1 + mu  * rho
    elif x == 1 and y == 1: return 1 - rho
    else:                   return 1.0

def dc_log_like_decay(x, y, alpha_x, beta_x, alpha_y, beta_y, rho, gamma, t, xi=0.0):
    lam = np.exp(alpha_x + beta_y + gamma)
    mu  = np.exp(alpha_y + beta_x)
    if lam <= 0 or mu <= 0:
        return -1e10
    tau = rho_correction(x, y, lam, mu, rho)
    if tau <= 0:
        return -1e10
    ll = (np.log(tau) +
          np.log(poisson.pmf(x, lam) + 1e-10) +
          np.log(poisson.pmf(y, mu)  + 1e-10))
    return np.exp(-xi * t) * ll

def neg_log_likelihood(params, df, teams, xi=0.0):
    n = len(teams)
    log_attack = np.zeros(n)
    log_attack[1:] = params[:n - 1]
    log_defence    = params[n - 1: 2 * n - 1]
    rho   = params[2 * n - 1]
    gamma = params[2 * n]
    idx = {t: i for i, t in enumerate(teams)}
    total = 0.0
    for _, row in df.iterrows():
        h, a = row['HomeTeam'], row['AwayTeam']
        if h not in idx or a not in idx:
            continue
        i, j = idx[h], idx[a]
        total += dc_log_like_decay(
            row['HomeGoals'], row['AwayGoals'],
            log_attack[i], log_defence[i],
            log_attack[j], log_defence[j],
            rho, gamma, row['time_diff'], xi
        )
    return -total

def fit_dixon_coles(df, teams, xi=0.00325):
    n = len(teams)
    x0 = np.zeros(2 * n + 1)
    x0[2 * n - 1] = -0.1
    x0[2 * n]     =  0.3
    bounds = ([(None, None)] * (n - 1) +
              [(None, None)] * n +
              [(-2.0, 2.0)] +
              [(0.0,  2.0)])
    print(f"  Optimizando: {n} equipos, {len(df)} partidos, xi={xi}...")
    result = minimize(
        fun=neg_log_likelihood, x0=x0, args=(df, teams, xi),
        method='L-BFGS-B', bounds=bounds,
        options={'maxiter': 1000, 'ftol': 1e-9, 'gtol': 1e-6}
    )
    print(f"  Convergencia: {result.success} | Log-lik: {-result.fun:.2f}")
    p = result.x
    log_attack = np.zeros(n)
    log_attack[1:] = p[:n - 1]
    log_defence    = p[n - 1: 2 * n - 1]
    out = {}
    for i, team in enumerate(teams):
        out[f'attack_{team}']  = float(log_attack[i])
        out[f'defence_{team}'] = float(log_defence[i])
    out['rho']            = float(p[2 * n - 1])
    out['home_adv']       = float(p[2 * n])
    out['xi']             = xi
    out['converged']      = bool(result.success)
    out['log_likelihood'] = float(-result.fun)
    return out

def get_one_season_teams(df_train):
    team_count = {}
    for season, grp in df_train.groupby('Season'):
        for t in set(grp['HomeTeam'].unique()) | set(grp['AwayTeam'].unique()):
            team_count[t] = team_count.get(t, 0) + 1
    return sorted(t for t, c in team_count.items() if c == 1)

def assign_new_team_params(params_dict, new_teams, reference_teams):
    if not new_teams:
        return params_dict
    if not reference_teams:
        print("  ⚠ Sin equipos de referencia, usando media global.")
        reference_teams = [k.replace('attack_', '') for k in params_dict if k.startswith('attack_')]
    avg_attack  = float(np.mean([params_dict[f'attack_{t}']  for t in reference_teams]))
    avg_defence = float(np.mean([params_dict[f'defence_{t}'] for t in reference_teams]))
    print(f"\n  Equipos de referencia (de paso): {reference_teams}")
    print(f"  Media ataque log={avg_attack:.4f}  |  media defensa log={avg_defence:.4f}")
    for team in new_teams:
        params_dict[f'attack_{team}']  = avg_attack
        params_dict[f'defence_{team}'] = avg_defence
        print(f"  → {team}: ataque={avg_attack:.4f}, defensa={avg_defence:.4f}")
    return params_dict

def run(train_seasons, test_seasons, output_file, xi=0.00325):
    print("\n" + "="*60)
    print(f"ENTRENAMIENTO : {train_seasons}")
    print(f"TEST          : {test_seasons}")
    print(f"OUTPUT        : {output_file}")
    print("="*60)

    print("\n[1/5] Descargando temporadas de entrenamiento...")
    df_train = prepare_data(load_seasons(train_seasons))

    print("\n[2/5] Descargando temporadas de test...")
    df_test = prepare_data(load_seasons(test_seasons))

    train_teams = sorted(set(df_train['HomeTeam'].unique()) | set(df_train['AwayTeam'].unique()))
    test_teams  = sorted(set(df_test['HomeTeam'].unique())  | set(df_test['AwayTeam'].unique()))
    new_teams   = sorted(set(test_teams) - set(train_teams))

    print(f"\n  Equipos en train : {len(train_teams)}")
    print(f"  Equipos en test  : {len(test_teams)}")
    print(f"  Equipos nuevos   : {new_teams}")

    max_date = df_train['Date'].max()
    df_train['time_diff'] = (max_date - df_train['Date']).dt.days

    print("\n[3/5] Ajustando modelo Dixon-Coles...")
    params = fit_dixon_coles(df_train, train_teams, xi=xi)

    print("\n[4/5] Identificando equipos de paso (1 sola temporada en train)...")
    one_season_teams = get_one_season_teams(df_train)
    print(f"  Equipos de paso: {one_season_teams}")

    print("\n[5/5] Asignando parámetros a equipos nuevos...")
    params = assign_new_team_params(params, new_teams, one_season_teams)

    all_teams = sorted(set(train_teams) | set(test_teams))
    rows = []
    for team in all_teams:
        if f'attack_{team}' in params:
            rows.append({
                'team'       : team,
                'log_attack' : round(params[f'attack_{team}'],  4),
                'attack'     : round(np.exp(params[f'attack_{team}']), 4),
                'log_defence': round(params[f'defence_{team}'], 4),
                'is_new'     : team in new_teams,
            })
    df_table = pd.DataFrame(rows).sort_values('attack', ascending=False).reset_index(drop=True)
    print("\n─── Parámetros (ordenados por ataque) ───")
    print(df_table.to_string(index=False))

    output = {
        "meta": {
            "train_seasons"           : train_seasons,
            "test_seasons"            : test_seasons,
            "xi"                      : xi,
            "rho"                     : round(params['rho'],      4),
            "home_adv"                : round(params['home_adv'], 4),
            "converged"               : params['converged'],
            "log_likelihood"          : round(params['log_likelihood'], 2),
            "new_teams"               : new_teams,
            "reference_teams_for_new" : one_season_teams,
        },
        "teams": {
            row['team']: {
                "log_attack" : row['log_attack'],
                "attack"     : row['attack'],
                "log_defence": row['log_defence'],
                "is_new"     : row['is_new'],
            }
            for row in df_table.to_dict('records')
        }
    }
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"\n✅ JSON guardado en: {output_file}")
    return output

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIG  ← edita aquí para cambiar temporadas
# ═══════════════════════════════════════════════════════════════
TRAIN_SEASONS = ['2021', '2122', '2223', '2324']
TEST_SEASONS  = ['2425']
OUTPUT_FILE   = 'estimaciones_2324.json'
XI            = 0.00325
# ═══════════════════════════════════════════════════════════════

# ── EJECUTAR ──────────────────────────────────────────────────
result = run(TRAIN_SEASONS, TEST_SEASONS, OUTPUT_FILE, XI)


ENTRENAMIENTO : ['2021', '2122', '2223', '2324']
TEST          : ['2425']
OUTPUT        : estimaciones_2324.json

[1/5] Descargando temporadas de entrenamiento...
  ✓ Temporada 2021: 380 partidos
  ✓ Temporada 2122: 380 partidos
  ✓ Temporada 2223: 380 partidos
  ✓ Temporada 2324: 380 partidos

[2/5] Descargando temporadas de test...
  ✓ Temporada 2425: 380 partidos

  Equipos en train : 26
  Equipos en test  : 20
  Equipos nuevos   : ['Ipswich']

[3/5] Ajustando modelo Dixon-Coles...
  Optimizando: 26 equipos, 1520 partidos, xi=0.00325...
  Convergencia: True | Log-lik: -1096.66

[4/5] Identificando equipos de paso (1 sola temporada en train)...
  Equipos de paso: ['Luton', 'Norwich', 'Watford', 'West Brom']

[5/5] Asignando parámetros a equipos nuevos...

  Equipos de referencia (de paso): ['Luton', 'Norwich', 'Watford', 'West Brom']
  Media ataque log=-0.9127  |  media defensa log=1.0329
  → Ipswich: ataque=-0.9127, defensa=1.0329

─── Parámetros (ordenados por ataque) ───
        

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIG  ← edita aquí para cambiar temporadas
# ═══════════════════════════════════════════════════════════════
TRAIN_SEASONS = ['2021', '2122', '2223', '2324', '2425']
TEST_SEASONS  = ['2526']
OUTPUT_FILE   = 'estimaciones_2425.json'
XI            = 0.00325
# ═══════════════════════════════════════════════════════════════

# ── EJECUTAR ──────────────────────────────────────────────────
result = run(TRAIN_SEASONS, TEST_SEASONS, OUTPUT_FILE, XI)


ENTRENAMIENTO : ['2021', '2122', '2223', '2324', '2425']
TEST          : ['2526']
OUTPUT        : estimaciones_2425.json

[1/5] Descargando temporadas de entrenamiento...
  ✓ Temporada 2021: 380 partidos
  ✓ Temporada 2122: 380 partidos
  ✓ Temporada 2223: 380 partidos
  ✓ Temporada 2324: 380 partidos
  ✓ Temporada 2425: 380 partidos

[2/5] Descargando temporadas de test...
  ✓ Temporada 2526: 300 partidos

  Equipos en train : 27
  Equipos en test  : 20
  Equipos nuevos   : ['Sunderland']

[3/5] Ajustando modelo Dixon-Coles...
  Optimizando: 27 equipos, 1900 partidos, xi=0.00325...
  Convergencia: True | Log-lik: -1060.06

[4/5] Identificando equipos de paso (1 sola temporada en train)...
  Equipos de paso: ['Ipswich', 'Luton', 'Norwich', 'Watford', 'West Brom']

[5/5] Asignando parámetros a equipos nuevos...

  Equipos de referencia (de paso): ['Ipswich', 'Luton', 'Norwich', 'Watford', 'West Brom']
  Media ataque log=-0.7517  |  media defensa log=0.9770
  → Sunderland: ataque=-0.751